In [3]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
from pathlib import Path

# 1. Locate the three Excel files in data/raw
raw_dir = Path("../data/raw")
files = {
    "stunting": raw_dir / "jme_expand_database_stunting_2025.xlsx",
    "wasting": raw_dir / "jme_expand_database_wasting_2025.xlsx",
    "severe_wasting": raw_dir / "jme_expand_database_severe_wasting_2025.xlsx",
}

# 2. Load only the Trend sheet from each workbook
def load_trend_sheet(path: Path, skiprows: int = 6) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name="Trend", skiprows=skiprows)
    df.columns = df.columns.astype(str).str.strip()
    return df

def pick_column(df: pd.DataFrame, candidates: list[str], contains: list[str] | None = None) -> str:
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for name in candidates:
        key = name.lower()
        if key in lower_map:
            return lower_map[key]

    if contains:
        for c in df.columns:
            c_lower = str(c).strip().lower()
            if any(token in c_lower for token in contains):
                return c

    raise KeyError(f"None of {candidates} found in columns: {list(df.columns)}")

def filter_year(df: pd.DataFrame, year_col: str, target_year: int) -> pd.DataFrame:
    years = pd.to_numeric(df[year_col], errors="coerce")
    if (years == target_year).any():
        return df[years == target_year]

    latest_year = int(years.max())
    print(f"Requested year {target_year} not found. Using latest available year: {latest_year}")
    return df[years == latest_year]

df_stunting = load_trend_sheet(files["stunting"])
df_wasting = load_trend_sheet(files["wasting"])
df_severe = load_trend_sheet(files["severe_wasting"])

# 3. Resolve key column names (robust to naming differences)
year_stunting = pick_column(df_stunting, ["Year", "Year*", "Time Period", "Time period"], contains=["year", "time"])

year_wasting = pick_column(df_wasting, ["Year", "Year*", "Time Period", "Time period"], contains=["year", "time"])

year_severe = pick_column(df_severe, ["Year", "Year*", "Time Period", "Time period"], contains=["year", "time"])

iso_stunting = pick_column(df_stunting, ["ISO3", "ISO", "ISO Code"], contains=["iso"])

iso_wasting = pick_column(df_wasting, ["ISO3", "ISO", "ISO Code"], contains=["iso"])

iso_severe = pick_column(df_severe, ["ISO3", "ISO", "ISO Code"], contains=["iso"])

country_stunting = pick_column(
    df_stunting,
    ["Country and areas", "Countries and areas", "Country", "Country Name"],
    contains=["country"],
)
estimate_stunting = pick_column(
    df_stunting,
    ["Point Estimate", "National", "Latest Estimate", "Estimate", "Value"],
    contains=["national", "estimate", "value"],
)
estimate_wasting = pick_column(
    df_wasting,
    ["Point Estimate", "National", "Latest Estimate", "Estimate", "Value"],
    contains=["national", "estimate", "value"],
)
estimate_severe = pick_column(
    df_severe,
    ["Point Estimate", "National", "Latest Estimate", "Estimate", "Value"],
    contains=["national", "estimate", "value"],
)

# 4. Filter for the target year (or latest available year if missing)
target_year = 2024
df_stunting_recent = filter_year(df_stunting, year_stunting, target_year)
df_wasting_recent = filter_year(df_wasting, year_wasting, target_year)
df_severe_recent = filter_year(df_severe, year_severe, target_year)

# 5. Keep only needed columns, cast estimates to numeric, and rename for clarity
stunting_cols = df_stunting_recent[[iso_stunting, country_stunting, estimate_stunting]].rename(
    columns={
        iso_stunting: "ISO3",
        country_stunting: "Country and areas",
        estimate_stunting: "Point Estimate_stunting",
    }
)
wasting_cols = df_wasting_recent[[iso_wasting, estimate_wasting]].rename(
    columns={
        iso_wasting: "ISO3",
        estimate_wasting: "Point Estimate_wasting",
    }
)
severe_cols = df_severe_recent[[iso_severe, estimate_severe]].rename(
    columns={
        iso_severe: "ISO3",
        estimate_severe: "Point Estimate_severe_wasting",
    }
)

stunting_cols["Point Estimate_stunting"] = pd.to_numeric(stunting_cols["Point Estimate_stunting"], errors="coerce")
wasting_cols["Point Estimate_wasting"] = pd.to_numeric(wasting_cols["Point Estimate_wasting"], errors="coerce")
severe_cols["Point Estimate_severe_wasting"] = pd.to_numeric(severe_cols["Point Estimate_severe_wasting"], errors="coerce")

# 6. Merge into a single master table
master_df = stunting_cols.merge(wasting_cols, on="ISO3", how="inner")
master_df = master_df.merge(severe_cols, on="ISO3", how="inner")
master_df = master_df.dropna(subset=["Point Estimate_stunting", "Point Estimate_wasting", "Point Estimate_severe_wasting"])

# 7. Preview
master_df.head()

,ISO3,Country and areas,Point Estimate_stunting,Point Estimate_wasting,Point Estimate_severe_wasting
0,BDI,Burundi,52.8,7.8,2.1
1,ECU,Ecuador,17.4,0.7,0.1
2,LSO,Lesotho,35.6,1.6,0.2
3,MLI,Mali,25.1,5.4,0.7
4,LKA,Sri Lanka,10.5,9.3,0.8


In [ ]:
# 8. Save the cleaned master dataset
output_path = Path("../data/processed/master_df_trend.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
master_df.to_csv(output_path, index=False)
print(f"Saved: {output_path.resolve()}")
print(f"Rows: {len(master_df)}, Columns: {master_df.shape[1]}")